In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
# the 3 raw files FinMark gave us
df_demo = pd.read_csv("customer_demographics_contaminated.csv")
df_trans = pd.read_csv("customer_transactions_contaminated.csv")
df_social = pd.read_csv("social_media_interactions_contaminated.csv")

print(df_demo.shape, df_trans.shape, df_social.shape)
df_demo.head()

(3200, 6) (3200, 6) (3200, 6)


,CustomerID,Age,Gender,Location,IncomeLevel,SignupDate
0,9207fa75-5758-48d1-94ad-19c041e0520f,51.0,Female,Jensenberg,Low,2022-11-17
1,5fb09cd8-a473-46f7-80bd-6e49cf509078,NaN,Female,Castilloport,High,2020-07-21
2,c139496e-cc89-498a-bd90-1fb4627b6cff,37.0,Male,Lake Jennifertown,NaN,2021-01-01
3,50118139-7264-428f-81cc-a25fddc5d6dd,44.0,Male,Port Carl,Medium,2024-06-10
4,7d1f2bbc-8d16-4fbc-9b37-ece3324e8ed4,50.0,Female,Jessebury,High,2023-08-24


In [3]:
# just eyeballing what we're working with first
df_demo.info()
df_trans.info()
df_social.info()

<class 'pandas.DataFrame'>
RangeIndex: 3200 entries, 0 to 3199
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   CustomerID   3200 non-null   str  
 1   Age          2909 non-null   str  
 2   Gender       3200 non-null   str  
 3   Location     3200 non-null   str  
 4   IncomeLevel  2897 non-null   str  
 5   SignupDate   3200 non-null   str  
dtypes: str(6)
memory usage: 150.1 KB
<class 'pandas.DataFrame'>
RangeIndex: 3200 entries, 0 to 3199
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   CustomerID       3200 non-null   str  
 1   TransactionID    3200 non-null   str  
 2   TransactionDate  3200 non-null   str  
 3   Amount           2896 non-null   str  
 4   ProductCategory  2901 non-null   str  
 5   PaymentMethod    3200 non-null   str  
dtypes: str(6)
memory usage: 150.1 KB
<class 'pandas.DataFrame'>
RangeIndex: 3200 entries, 0 to 3199

In [4]:
# how many exact duplicate rows are hiding in each file
print("dupes:", df_demo.duplicated().sum(), df_trans.duplicated().sum(), df_social.duplicated().sum())

dupes: 177 185 180


In [5]:
# where are the blanks
print(df_demo.isnull().sum())
print(df_trans.isnull().sum())
print(df_social.isnull().sum())

CustomerID       0
Age            291
Gender           0
Location         0
IncomeLevel    303
SignupDate       0
dtype: int64
CustomerID           0
TransactionID        0
TransactionDate      0
Amount             304
ProductCategory    299
PaymentMethod        0
dtype: int64
CustomerID           0
InteractionID        0
InteractionDate      0
Platform           311
InteractionType      0
Sentiment          329
dtype: int64


In [6]:
# Age has "Unknown" typed in instead of just being blank, and Amount has "Free" instead of 0 - caught these while scrolling through
print(df_demo['Age'].unique()[:5])
print(df_trans['Amount'].unique()[:5])

<StringArray>
['51.0', nan, '37.0', '44.0', '50.0']
Length: 5, dtype: str
<StringArray>
['117.64', '466.14', '563.99', '254.44', '590.52']
Length: 5, dtype: str


In [7]:
# dates are inconsistent too - some rows are YYYY-MM-DD, others are DD/MM/YYYY
# wrote this so we can parse both without losing rows
def fix_dates(col):
    def try_parse(val):
        if pd.isna(val):
            return pd.NaT
        try:
            return pd.to_datetime(val, format='%Y-%m-%d')
        except (ValueError, TypeError):
            pass
        try:
            return pd.to_datetime(val, format='%d/%m/%Y')
        except (ValueError, TypeError):
            return pd.NaT
    return col.apply(try_parse)

In [8]:
# ---------- demographics ----------
demo_clean = df_demo.copy()
demo_clean['Age'] = pd.to_numeric(demo_clean['Age'].replace('Unknown', np.nan), errors='coerce')
demo_clean['Age'] = demo_clean['Age'].fillna(demo_clean['Age'].median()).astype(int)  # median so outliers don't skew it
demo_clean['IncomeLevel'] = demo_clean['IncomeLevel'].fillna('Unknown')  # don't want to guess someone's income bracket
demo_clean['SignupDate'] = fix_dates(demo_clean['SignupDate'])

for col in ['Gender', 'Location', 'IncomeLevel']:
    demo_clean[col] = demo_clean[col].str.strip().str.title()  # inconsistent spacing/casing was everywhere

demo_clean = demo_clean.drop_duplicates()

print(demo_clean.isnull().sum().sum(), demo_clean.duplicated().sum())
demo_clean.head()

0 0


,CustomerID,Age,Gender,Location,IncomeLevel,SignupDate
0,9207fa75-5758-48d1-94ad-19c041e0520f,51,Female,Jensenberg,Low,2022-11-17
1,5fb09cd8-a473-46f7-80bd-6e49cf509078,45,Female,Castilloport,High,2020-07-21
2,c139496e-cc89-498a-bd90-1fb4627b6cff,37,Male,Lake Jennifertown,Unknown,2021-01-01
3,50118139-7264-428f-81cc-a25fddc5d6dd,44,Male,Port Carl,Medium,2024-06-10
4,7d1f2bbc-8d16-4fbc-9b37-ece3324e8ed4,50,Female,Jessebury,High,2023-08-24


In [9]:
# ---------- transactions ----------
trans_clean = df_trans.copy()
trans_clean['Amount'] = pd.to_numeric(trans_clean['Amount'].replace('Free', 0), errors='coerce')
trans_clean['Amount'] = trans_clean['Amount'].fillna(trans_clean['Amount'].median())
trans_clean['Amount'] = trans_clean['Amount'].abs().round(2)  # negative purchase amount makes no sense, no refund column to explain it either
trans_clean['ProductCategory'] = trans_clean['ProductCategory'].fillna('Unknown')
trans_clean['TransactionDate'] = fix_dates(trans_clean['TransactionDate'])

trans_clean = trans_clean.drop_duplicates().drop_duplicates(subset='TransactionID', keep='first')  # some rows had same TransactionID but slightly different values

print(trans_clean.isnull().sum().sum(), trans_clean.duplicated().sum())
trans_clean.head()

0 0


,CustomerID,TransactionID,TransactionDate,Amount,ProductCategory,PaymentMethod
0,60567026-f719-4cd6-849e-137e86d8938f,5ff75116-0a50-4d04-80fb-31e5ccbb0769,2024-05-15,117.64,Clothing,PayPal
1,4090ba85-b111-4f75-a792-c777965f5255,2c39b9fe-ff57-4d39-9321-9f5cdf187aa1,2023-04-26,466.14,Health & Beauty,Bank Transfer
2,9223891b-73ff-4d5c-b8ae-13ece82ee28b,f79588dd-3db9-4ffa-97f8-7de0e64259f1,2022-09-23,563.99,Clothing,Debit Card
3,9243eebc-938f-480c-8564-16d503d250de,401c0fc9-60df-4455-ad78-67c132f9897d,2024-04-15,254.44,Automotive,PayPal
4,6e3e8eb8-bc0f-4ffe-9f74-5d5efec9502f,2034aebc-8280-4254-a667-92bcd1c2be4f,2024-06-03,590.52,Home & Garden,Bank Transfer


In [10]:
# ---------- social media ----------
social_clean = df_social.copy()
social_clean['Platform'] = social_clean['Platform'].fillna('Unknown')
social_clean['Sentiment'] = social_clean['Sentiment'].fillna('Not Specified')  # didn't want to default this to "Neutral" since that's an actual sentiment value
social_clean['InteractionDate'] = fix_dates(social_clean['InteractionDate'])

social_clean = social_clean.drop_duplicates().drop_duplicates(subset='InteractionID', keep='first')

print(social_clean.isnull().sum().sum(), social_clean.duplicated().sum())
social_clean.head()

0 0


,CustomerID,InteractionID,InteractionDate,Platform,InteractionType,Sentiment
0,2dcb9523-356b-40b2-a67b-1f27797de261,e5d15761-d0a7-4329-89e3-79a892c56097,2023-07-11,Unknown,Comment,Not Specified
1,e12c37b3-7d4d-472f-9fd8-0df2cb3001aa,02f9f376-70ae-4fcd-9070-1db977939948,2023-07-06,Twitter,Share,Not Specified
2,08a911a3-65e6-4f5d-a6a1-ae7ddcbe28a2,a83fa04c-f109-4f24-8ce1-2078154f6a1c,2024-05-24,Instagram,Comment,Neutral
3,efdfdfc9-5dbb-4478-911a-101a390a0285,28a69c4b-a2e4-4c74-a130-1132d7733fdf,2023-11-01,Instagram,Like,Neutral
4,ca1e90f6-0e5f-492e-ab92-252ff540da18,d9d1c6f8-5e15-4738-b52b-13c2982420cc,2023-07-08,Instagram,Like,Not Specified


In [11]:
# before/after numbers for the report
summary = pd.DataFrame({
    'Dataset': ['Demographics', 'Transactions', 'Social Media'],
    'Rows Before': [len(df_demo), len(df_trans), len(df_social)],
    'Rows After': [len(demo_clean), len(trans_clean), len(social_clean)],
    'Dupes Before': [df_demo.duplicated().sum(), df_trans.duplicated().sum(), df_social.duplicated().sum()],
    'Nulls Before': [df_demo.isnull().sum().sum(), df_trans.isnull().sum().sum(), df_social.isnull().sum().sum()],
})
summary

,Dataset,Rows Before,Rows After,Dupes Before,Nulls Before
0,Demographics,3200,3023,177,594
1,Transactions,3200,3000,185,603
2,Social Media,3200,3000,180,640


In [12]:
# export the final versions for github
demo_clean.to_csv('customer_demographics_cleaned.csv', index=False)
trans_clean.to_csv('customer_transactions_cleaned.csv', index=False)
social_clean.to_csv('social_media_interactions_cleaned.csv', index=False)

print("saved all 3 cleaned files")

saved all 3 cleaned files
